# Chapter 1 subtitle PNG sequence exporter

This notebook mirrors the sentence parsing logic from `src/StoryHorizontal.jsx` for chapter 1 and renders transparent subtitle PNGs with Story-style ending colors. It also expands those sentence cards into a frame-numbered PNG sequence timed from `words_and_start_times.js`.


In [ ]:
from __future__ import annotations

import json
import math
import re
import shutil
from pathlib import Path
from statistics import median

from PIL import Image, ImageDraw, ImageFont

# Hard-coded paths for the first experiment.
ROOT = Path(r"c:/Users/Svante/Hacking/navigation-test-stylish")
CHAPTER_DIR = ROOT / "src" / "story" / "chapter1"
STORY_TEXT_PATH = CHAPTER_DIR / "story_text.js"
TIMES_PATH = CHAPTER_DIR / "words_and_start_times.js"
OUTPUT_DIR = CHAPTER_DIR / "subtitles_experimental"
CARDS_DIR = OUTPUT_DIR / "sentence_cards"
SEQUENCE_DIR = OUTPUT_DIR / "png_sequence"
PREVIEW_DIR = OUTPUT_DIR / "sentence_cards_preview_white"
MANIFEST_PATH = OUTPUT_DIR / "timing_manifest.json"
FONT_PATH = Path(r"C:/Windows/Fonts/seguisb.ttf")

# Render settings.
FPS = 30
FRAME_WIDTH = 1920
SHOW_SPOKEN_FINNISH = False
FRAME_HEIGHT = 1080
LEFT_MARGIN = 120
WORD_GAP = 26
PRIMARY_FONT_SIZE = 74
SECONDARY_FONT_SIZE = 42
ENGLISH_FONT_SIZE = 42
SPOKEN_FONT_SIZE = 36
BOTTOM_MARGIN = 84
ROW_GAP = 24
PRIMARY_STROKE_WIDTH = 6
SECONDARY_STROKE_WIDTH = 4

# Colors aligned to StoryHorizontal / Story.css, but slightly softer for video.
COLOR_DEFAULT = (70, 76, 86, 255)
COLOR_MUTED = (70, 76, 86, 245)
COLOR_ENGLISH = (70, 76, 86, 245)
COLOR_SPOKEN = (70, 76, 86, 238)
COLOR_ENDING1 = (110, 231, 255, 242)
COLOR_ENDING2 = (74, 222, 128, 242)
COLOR_STROKE = (255, 255, 255, 215)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CARDS_DIR.mkdir(parents=True, exist_ok=True)
SEQUENCE_DIR.mkdir(parents=True, exist_ok=True)
PREVIEW_DIR.mkdir(parents=True, exist_ok=True)
for existing_png in CARDS_DIR.glob("*.png"):
    existing_png.unlink()
for existing_png in SEQUENCE_DIR.glob("*.png"):
    existing_png.unlink()
for existing_png in PREVIEW_DIR.glob("*.png"):
    existing_png.unlink()

assert STORY_TEXT_PATH.exists(), STORY_TEXT_PATH
assert TIMES_PATH.exists(), TIMES_PATH
assert FONT_PATH.exists(), FONT_PATH

story_text_source = STORY_TEXT_PATH.read_text(encoding="utf-8")
times_source = TIMES_PATH.read_text(encoding="utf-8")
primary_font = ImageFont.truetype(str(FONT_PATH), PRIMARY_FONT_SIZE)
secondary_font = ImageFont.truetype(str(FONT_PATH), SECONDARY_FONT_SIZE)
english_font = ImageFont.truetype(str(FONT_PATH), ENGLISH_FONT_SIZE)
spoken_font = ImageFont.truetype(str(FONT_PATH), SPOKEN_FONT_SIZE)

primary_height = PRIMARY_FONT_SIZE + PRIMARY_STROKE_WIDTH * 2
secondary_height = SECONDARY_FONT_SIZE + SECONDARY_STROKE_WIDTH * 2
english_height = ENGLISH_FONT_SIZE + SECONDARY_STROKE_WIDTH * 2
spoken_height = SPOKEN_FONT_SIZE + SECONDARY_STROKE_WIDTH * 2
visible_row_count = 3 + (1 if SHOW_SPOKEN_FINNISH else 0)
stack_height = primary_height + secondary_height + english_height + (spoken_height if SHOW_SPOKEN_FINNISH else 0) + ROW_GAP * (visible_row_count - 1)
stack_top_y = FRAME_HEIGHT - BOTTOM_MARGIN - stack_height
PRIMARY_Y = stack_top_y
FAKE_ENGLISH_Y = PRIMARY_Y + primary_height + ROW_GAP
ENGLISH_Y = FAKE_ENGLISH_Y + secondary_height + ROW_GAP
SPOKEN_Y = ENGLISH_Y + english_height + ROW_GAP


In [ ]:
def extract_template_literal(source: str, const_name: str) -> str:
    pattern = rf"const\s+{re.escape(const_name)}\s*=\s*`([\s\S]*?)`;"
    match = re.search(pattern, source)
    if not match:
        raise ValueError(f"Could not find template literal for {const_name}")
    return match.group(1)


def extract_string_array(source: str, const_name: str) -> list[str]:
    pattern = rf"export\s+const\s+{re.escape(const_name)}\s*=\s*\[(.*?)\];"
    match = re.search(pattern, source, re.S)
    if not match:
        raise ValueError(f"Could not find array for {const_name}")
    body = match.group(1)
    return [bytes(item, "utf-8").decode("unicode_escape") for item in re.findall(r'"((?:[^"\\]|\\.)*)"', body)]


def extract_number_array(source: str, const_name: str) -> list[float]:
    pattern = rf"export\s+const\s+{re.escape(const_name)}\s*=\s*\[(.*?)\];"
    match = re.search(pattern, source, re.S)
    if not match:
        raise ValueError(f"Could not find array for {const_name}")
    return [float(item) for item in re.findall(r"-?\d+(?:\.\d+)?", match.group(1))]


def sanitize_token(token: str = "") -> str:
    return re.sub(r"\s+", " ", token).strip()


def build_word_entry(word1: str, word2: str, word3: str, word4: str) -> dict:
    raw1 = sanitize_token(word1)
    raw2 = sanitize_token(word2)
    raw3 = sanitize_token(word3)
    raw4 = sanitize_token(word4)

    is_ending1 = raw1.startswith("-")
    is_ending2 = raw1.startswith("==") or raw1.startswith("=")

    display1 = raw1
    display2 = raw2
    display4 = raw4

    if is_ending1:
        display1 = raw1[1:]
        display4 = raw4[1:] if raw4.startswith("-") else raw4

    if is_ending2:
        display1 = re.sub(r"^=+", "", raw1)
        display4 = re.sub(r"^=+", "", raw4)
        display2 = raw2.replace("=", "-")

    return {
        "display1": display1,
        "display2": display2,
        "display3": raw3,
        "display4": display4,
        "isEnding1": is_ending1,
        "isEnding2": is_ending2,
    }


def ends_sentence(token: str) -> bool:
    return bool(re.search(r"[.!?][\"”']?$", token))


story_finnish = extract_template_literal(story_text_source, "storyFinnish")
story_fake_english = extract_template_literal(story_text_source, "storyFakeEnglish")
story_english = extract_template_literal(story_text_source, "storyEnglish")
story_spoken_finnish = extract_template_literal(story_text_source, "storySpokenFinnish")

highlight_words = story_finnish.split(" ")
highlight_words2 = story_fake_english.split(" ")
highlight_words3 = story_english.split(" ")
highlight_words4 = story_spoken_finnish.split(" ")

word_entries = [
    build_word_entry(highlight_words[idx], highlight_words2[idx], highlight_words3[idx], highlight_words4[idx])
    for idx in range(min(len(highlight_words), len(highlight_words2), len(highlight_words4)))
]

sentence_groups = []
current_group = []
for idx, word in enumerate(word_entries):
    if not word["display1"] and not word["display2"] and not word["display4"]:
        continue
    current_group.append(idx)
    if ends_sentence(word["display1"]):
        sentence_groups.append(current_group)
        current_group = []
if current_group:
    sentence_groups.append(current_group)

english_words = [word for word in highlight_words3 if word.strip() != ""]
english_sentence_groups = []
current_english_group = []
for idx, word in enumerate(english_words):
    current_english_group.append(idx)
    if ends_sentence(word):
        english_sentence_groups.append(current_english_group)
        current_english_group = []
if current_english_group:
    english_sentence_groups.append(current_english_group)

timed_words = extract_string_array(times_source, "words")
start_times = extract_number_array(times_source, "startTimes")

assert len(timed_words) == len(start_times), "words and startTimes length mismatch"

def find_word_number_for_index(word_index: int) -> int:
    count = 0
    for idx in range(0, min(word_index + 1, len(word_entries))):
        if word_entries[idx]["isEnding1"] or word_entries[idx]["isEnding2"]:
            continue
        count += 1
    return max(1, count)


def compose_sentence_text(indexes: list[int], display_key: str) -> str:
    parts = []
    for idx in indexes:
        word = word_entries[idx]
        text = word[display_key]
        if not text:
            continue
        if word["isEnding1"] or word["isEnding2"]:
            if parts:
                parts[-1] = parts[-1] + text
            else:
                parts.append(text)
        else:
            parts.append(text)
    return " ".join(parts)


def compose_english_sentence(indexes: list[int]) -> str:
    return " ".join(english_words[idx].strip() for idx in indexes if english_words[idx].strip())


def estimate_last_duration(values: list[float]) -> float:
    if len(values) < 2:
        return 1.5
    diffs = [round(values[idx + 1] - values[idx], 3) for idx in range(len(values) - 1) if values[idx + 1] > values[idx]]
    if not diffs:
        return 1.5
    return max(0.7, min(2.0, median(diffs) * 2.6))


last_sentence_duration = estimate_last_duration(start_times)
sentence_records = []
for sentence_index, indexes in enumerate(sentence_groups):
    base_indexes = [idx for idx in indexes if not word_entries[idx]["isEnding1"] and not word_entries[idx]["isEnding2"]]
    first_word_number = find_word_number_for_index(base_indexes[0])
    start_time = start_times[first_word_number - 1]
    if sentence_index + 1 < len(sentence_groups):
        next_base_indexes = [
            idx for idx in sentence_groups[sentence_index + 1]
            if not word_entries[idx]["isEnding1"] and not word_entries[idx]["isEnding2"]
        ]
        next_word_number = find_word_number_for_index(next_base_indexes[0])
        end_time = start_times[next_word_number - 1]
    else:
        end_time = start_times[-1] + last_sentence_duration
    english_indexes = english_sentence_groups[sentence_index] if sentence_index < len(english_sentence_groups) else []
    sentence_records.append({
        "sentence_index": sentence_index,
        "indexes": indexes,
        "english_indexes": english_indexes,
        "start_time": start_time,
        "end_time": end_time,
    })

len(sentence_records), sentence_records[0]


In [ ]:
def token_advance(draw: ImageDraw.ImageDraw, text: str, font: ImageFont.FreeTypeFont) -> float:
    return float(draw.textlength(text, font=font))


def aligned_row_width(
    draw: ImageDraw.ImageDraw,
    indexes: list[int],
    display_key: str,
    font: ImageFont.FreeTypeFont,
) -> float:
    width = 0.0
    first_visible = True
    for idx in indexes:
        word = word_entries[idx]
        text = word[display_key]
        if not text:
            continue
        if not word["isEnding1"] and not word["isEnding2"] and not first_visible:
            width += WORD_GAP
        width += token_advance(draw, text, font)
        first_visible = False
    return width


def plain_row_width(
    draw: ImageDraw.ImageDraw,
    words: list[str],
    font: ImageFont.FreeTypeFont,
) -> float:
    width = 0.0
    visible_count = 0
    for word in words:
        text = word.strip()
        if not text:
            continue
        if visible_count > 0:
            width += WORD_GAP
        width += token_advance(draw, text, font)
        visible_count += 1
    return width


def centered_x(row_width: float) -> float:
    usable_width = FRAME_WIDTH - (LEFT_MARGIN * 2)
    if row_width >= usable_width:
        return float(LEFT_MARGIN)
    return float((FRAME_WIDTH - row_width) / 2)


def render_aligned_row(
    draw: ImageDraw.ImageDraw,
    indexes: list[int],
    display_key: str,
    font: ImageFont.FreeTypeFont,
    y: int,
    default_color: tuple[int, int, int, int],
    stroke_width: int,
    show_ending_colors: bool = True,
) -> None:
    row_width = aligned_row_width(draw, indexes, display_key, font)
    x = centered_x(row_width)
    first_visible = True
    for idx in indexes:
        word = word_entries[idx]
        text = word[display_key]
        if not text:
            continue
        if not word["isEnding1"] and not word["isEnding2"] and not first_visible:
            x += WORD_GAP
        color = default_color
        if show_ending_colors:
            if word["isEnding1"]:
                color = COLOR_ENDING1
            elif word["isEnding2"]:
                color = COLOR_ENDING2
        draw.text(
            (x, y),
            text,
            font=font,
            fill=color,
            stroke_width=stroke_width,
            stroke_fill=COLOR_STROKE,
        )
        x += token_advance(draw, text, font)
        first_visible = False


def render_plain_row(
    draw: ImageDraw.ImageDraw,
    words: list[str],
    font: ImageFont.FreeTypeFont,
    y: int,
    color: tuple[int, int, int, int],
    stroke_width: int,
) -> None:
    row_width = plain_row_width(draw, words, font)
    x = centered_x(row_width)
    for idx, word in enumerate(words):
        text = word.strip()
        if not text:
            continue
        if idx > 0:
            x += WORD_GAP
        draw.text(
            (x, y),
            text,
            font=font,
            fill=color,
            stroke_width=stroke_width,
            stroke_fill=COLOR_STROKE,
        )
        x += token_advance(draw, text, font)


def export_preview_on_white(source_path: Path) -> None:
    transparent = Image.open(source_path).convert("RGBA")
    preview = Image.new("RGBA", transparent.size, (255, 255, 255, 255))
    preview.alpha_composite(transparent)
    preview.save(PREVIEW_DIR / source_path.name)


def render_sentence_card(record: dict) -> Path:
    image = Image.new("RGBA", (FRAME_WIDTH, FRAME_HEIGHT), (0, 0, 0, 0))
    draw = ImageDraw.Draw(image)
    render_aligned_row(draw, record["indexes"], "display1", primary_font, PRIMARY_Y, COLOR_DEFAULT, PRIMARY_STROKE_WIDTH)
    render_aligned_row(draw, record["indexes"], "display2", secondary_font, FAKE_ENGLISH_Y, COLOR_MUTED, SECONDARY_STROKE_WIDTH)
    english_row_words = [english_words[idx] for idx in record["english_indexes"]]
    render_plain_row(draw, english_row_words, english_font, ENGLISH_Y, COLOR_ENGLISH, SECONDARY_STROKE_WIDTH)
    if SHOW_SPOKEN_FINNISH:
        render_aligned_row(draw, record["indexes"], "display4", spoken_font, SPOKEN_Y, COLOR_SPOKEN, SECONDARY_STROKE_WIDTH)

    card_path = CARDS_DIR / f"sentence_{record['sentence_index'] + 1:03d}.png"
    image.save(card_path)
    export_preview_on_white(card_path)
    return card_path


card_paths = []
for record in sentence_records:
    card_paths.append(render_sentence_card(record))

total_frames = math.ceil(sentence_records[-1]["end_time"] * FPS)
manifest = []
start_frames = [round(record["start_time"] * FPS) for record in sentence_records]

for record, card_path in zip(sentence_records, card_paths):
    start_frame = start_frames[record["sentence_index"]]
    if record["sentence_index"] + 1 < len(sentence_records):
        end_frame = max(start_frame + 1, start_frames[record["sentence_index"] + 1])
    else:
        end_frame = max(start_frame + 1, total_frames)
    manifest.append({
        "sentence_index": record["sentence_index"] + 1,
        "text_finnish": compose_sentence_text(record["indexes"], "display1"),
        "text_fake_english": compose_sentence_text(record["indexes"], "display2"),
        "text_english": compose_english_sentence(record["english_indexes"]),
        "text_spoken_finnish": compose_sentence_text(record["indexes"], "display4"),
        "card": str(card_path.relative_to(ROOT)).replace("\\", "/"),
        "start_time": round(record["start_time"], 3),
        "end_time": round(record["end_time"], 3),
        "start_frame": start_frame,
        "end_frame_exclusive": end_frame,
    })
    for frame in range(start_frame, end_frame):
        frame_path = SEQUENCE_DIR / f"subtitle_{frame + 1:05d}.png"
        shutil.copyfile(card_path, frame_path)

with MANIFEST_PATH.open("w", encoding="utf-8") as handle:
    json.dump(
        {
            "fps": FPS,
            "frame_size": [FRAME_WIDTH, FRAME_HEIGHT],
            "show_spoken_finnish": SHOW_SPOKEN_FINNISH,
            "sequence_frame_count": total_frames,
            "sentence_count": len(sentence_records),
            "sentences": manifest,
        },
        handle,
        ensure_ascii=False,
        indent=2,
    )

manifest[:3], total_frames
